# Topic Modelling and Emotion Analysis of Customer Reviews

Reviews of a UK wellness centre chain from two public review platforms (Platform A, Platform B). English-language filtering and anonymisation, word frequencies, BERTopic on negative reviews, location-level comparison, BERT emotion classification, BERTopic on anger reviews, an instruction-tuned LLM for topic extraction and suggested actions, and Gensim LDA.

*Origin: University of Cambridge Data Science Career Accelerator coursework (2024-25). Data source anonymised and not included. Cell outputs cleared to protect the source data; results quoted in the README come from the original run.*

#### **Applying NLP for topic modelling in a real-life context**

### **1. Importing packages and data**

In [ ]:
### Importing packages and data

# Import packages
!pip install bertopic # for topic modelling
!pip install pyLDAvis # for visualising topic models
!pip install gspread
!pip install langdetect
!pip install transformers # for NLP models
!pip install spacy # for text processing
!python -m spacy download en_core_web_sm # downloads a small-sized English language model for SpaCy (en_core_web_sm)

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from langdetect import detect
from wordcloud import WordCloud
from bertopic import BERTopic
from transformers import pipeline
from gensim import corpora
from gensim.models import LdaModel
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
import umap
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
import torch
import plotly.express as px
import matplotlib.colors as mcolors
import plotly.io as pio # used for a diagnostic print
print(pio.renderers)
import logging
import re
import spacy

# Defining colour scheme for my visualisations
NAVY_BLUE = "#000080"
BLUE = "#0000FF"
WHITE = "#FFFFFF"
DARK_RED = "#8A0707"

# Import the data set with the provided URL:
# Data set's Drive link: data/reviews_file1.xlsx

# URLs for CSV export
platform_a_url = 'data/reviews_file2.xlsx'
platform_b_url = 'data/reviews_file3.xlsx'

# Criterion 1.1: Import the data file Platform A_12_months.xlsx into a dataframe
platform_a_df = pd.read_csv(platform_a_url)

# Criterion 1.2: Import the data file Platform B_12_months.xlsx into a dataframe
platform_b_df = pd.read_csv(platform_b_url)

# Criterion 1.3: Remove any rows with missing values in the Comment column (Platform A review) and Review Content column (Platform B)
platform_a_df = platform_a_df.dropna(subset=['Comment'])
platform_b_df = platform_b_df.dropna(subset=['Review Content'])

In [ ]:
# Converting to lowercase and stripping whitespace to enhance consistency for text processing
platform_a_df["Club's Name"] = platform_a_df["Club's Name"].str.strip().str.lower()
platform_b_df["Location Name"] = platform_b_df["Location Name"].str.strip().str.lower()

# Drop unnecessary platform_a coluns
platform_a_df = platform_a_df.drop(columns=['Customer Name', 'SurveyID for external use (e.g. tech support)', 'Social Media Source'])

# Drop unnecessary platform_b columns
platform_b_df = platform_b_df.drop(columns=['Review ID', 'Review Consumer User ID', 'Source Of Review', 'Review Language', 'Domain URL', 'Webshop Name', 'Business Unit ID', 'Tags', 'Location ID'], errors='ignore')

# Convert date columsn to datetime
platform_a_df['Creation Date'] = pd.to_datetime(platform_a_df['Creation Date'], errors='coerce')
platform_b_df['Review Created (UTC)'] = pd.to_datetime(platform_b_df['Review Created (UTC)'], errors='coerce')
platform_b_df['Company Reply Date (UTC)'] = pd.to_datetime(platform_b_df['Company Reply Date (UTC)'], errors='coerce')

# Display initial DataFrame info for verification
print("Platform A_df info:\n", platform_a_df.info())
print()
print("Platform B_df info:\n", platform_b_df.info())

In [ ]:
# Checking platform_a_df data set
print(platform_a_df.head())
print()
print(platform_a_df.info())
print()
print(platform_a_df.columns)

In [ ]:
# Checking platform_b_df data set
print(platform_b_df.head())
print()
print(platform_b_df.info())
print()
print(platform_b_df.columns)

 **-------------------------------------------------------------------- END OF SECTION----------------------------------------------------------------------- ################################################################################################################################################################################################################**

### **2. Conducting initial data investigation**

**For practicality, the analyses will be conducted solely on the English reviews**

In [ ]:
### Conducting initial data investigation

# Function to inspect dataframe info and missing values
def inspect_data(df, name):
    print(f"{name} Info:")
    print(df.info())
    print(f"{name} Missing Values:")
    print(df.isnull().sum())
    print()

# Inspect both data sets
inspect_data(platform_a_df, "Platform A")
inspect_data(platform_b_df, "Platform B")

# Function to filter english reviews using langdetect
# Disclosure: the function below was created with AI's assistance
def filter_english_reviews(df, text_column):
    def is_english(text):
        try:
            return detect(text) == 'en'
        except:
            return False
    return df[df[text_column].apply(is_english)]

# For practicality, the analyses will be conducted solely on the English reviews
# Filter for english reviews and replace original datarames
platform_a_df = filter_english_reviews(platform_a_df, 'Comment')
platform_b_df = filter_english_reviews(platform_b_df, 'Review Content')

# Validate filtered dataframes
print("Filtered platform_a dataframe (English only):")
print(platform_a_df.head())
print()
print("Filtered Platform B dataframe (English only):")
print(platform_b_df.head())
print()

In [ ]:
platform_a_df.info()

In [ ]:
platform_b_df.info()

In [ ]:
# Remove dduplicates after filtering to ensure unique reviews
platform_a_df = platform_a_df.drop_duplicates(subset=["Club's Name", 'Comment'])
platform_b_df = platform_b_df.drop_duplicates(subset=['Location Name', 'Review Content'])

In [ ]:
# Criterion 2.1: Find the number of unique locations in the Platform A data set
# Criterion 2.3: Use Club's Name for the Platform A data set
platform_a_unique_locations = platform_a_df["Club's Name"].nunique()

In [ ]:
# Criterion 2.2: Find the number of unique locations in the Platform B data set
# Criterion 2.4: Use Location Name for the Platform B data set
platform_b_unique_locations = platform_b_df['Location Name'].nunique()

In [ ]:
# Criterion 2.5: Find the number of common locations between the Platform A data set and the Platform B data set
common_locations = set(platform_a_df["Club's Name"]).intersection(set(platform_b_df['Location Name']))
print(f"Platform A Unique Locations: {platform_a_unique_locations}\n")
print(f"Platform B Unique Locations: {platform_b_unique_locations}\n")
print(f"Number of Common Locations: {len(common_locations)}\n")

The intersection of locations shows that not all locations are shared, implying potentially unmatched locations in both data sets.

In [ ]:
# Download NLTK resources for preprocessing
# NLTK’s preprocessing removes noise
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:
# Criterion 2.6: Perform preprocessing of the data – change to lower case, remove stopwords using NLTK, and remove numbers
# Criterion 2.7: Tokenise the data using word_tokenize from NLTK
stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    text = text.lower()  # Lowercase
    tokens = word_tokenize(text)  # tokenize
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]  # remove numbers and stopwords
    return ' '.join(tokens)

# Apply preprocessing to review text
platform_a_df['Comment'] = platform_a_df['Comment'].apply(preprocess_text)
platform_b_df['Review Content'] = platform_b_df['Review Content'].apply(preprocess_text)

In [ ]:
# Validate preprocessing
print("Platform A unique locations after preprocessing:", platform_a_df["Club's Name"].nunique())
print("Platform A total reviews:", len(platform_a_df))
print("Platform B unique locations after preprocessing:", platform_b_df['Location Name'].nunique())
print("Platform B total reviews:", len(platform_b_df))

In [ ]:
print(platform_a_df.head())
print()
print(platform_b_df.head())

In [ ]:
# Anonymisation of the data sets to remove any personal information

# Disclosure: The following function was created with AI's assistance
nlp = spacy.load("en_core_web_sm")

def anonymize_names_simple(df, text_columns):
    """
    Anonymize names in specified DataFrame columns using NER and a fallback name list,
    remove brand-related terms, and clean up extra spaces.

    Parameters:
    - df: pandas DataFrame containing the text data
    - text_columns: String or list of column names to anonymize

    Returns:
    - DataFrame with anonymized names and brand-related terms removed
    """
    # Ensure text_columns is a list
    if isinstance(text_columns, str):
        text_columns = [text_columns]
    if not all(col in df.columns for col in text_columns):
        raise ValueError(f"One or more columns {text_columns} not found in DataFrame")

    df_anon = df.copy()

    # Fallback list of common names (case-insensitive)
    common_names = {
        'ellie', 'kennedy', 'john', 'jane', 'mary', 'mari', 'smith', 'jones',
        'sarah', 'david', 'emma', 'michael', 'olivia', 'james', 'sophia', 'robert',
        'isabella', 'william', 'mia', 'daniel', 'ava', 'matthew', 'amelia', 'joseph'
    }
    name_pattern = r'\b(?:' + '|'.join(re.escape(name) for name in common_names) + r')\b'

    def replace_names_and_remove_brand(text):
        if not isinstance(text, str):
            return text

        # Step 1: Replace names using spaCy NER
        doc = nlp(text)
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                text = text.replace(ent.text, '[NAME]')

        # Step 2: Fallback - Replace known names from the list (case-insensitive)
        text = re.sub(name_pattern, '[NAME]', text, flags=re.IGNORECASE)

        # Step 3: Remove brand-related terms (actual brand terms withheld from the public version)
        brand_patterns = [
            r'\b<brand terms withheld>\b',
        ]
        combined_brand_pattern = '|'.join(brand_patterns)
        text = re.sub(combined_brand_pattern, '', text, flags=re.IGNORECASE)

        # Step 4: Clean up extra spaces
        text = re.sub(r'\s+', ' ', text).strip()

        return text

    # Apply the function to specified columns
    for col in text_columns:
        df_anon[col] = df_anon[col].apply(replace_names_and_remove_brand)
    return df_anon

# Implementation
platform_a_df_anon = anonymize_names_simple(platform_a_df, 'Comment')
platform_b_df_anon = anonymize_names_simple(platform_b_df, ['Review Title', 'Review Content'])

print("Anonymised Platform A DataFrame:")
print(platform_a_df_anon.head(20))
print("\nAnonymised Platform B DataFrame:")
print(platform_b_df_anon.head(20))

In [ ]:
# View results
print("Anonymised Platform A DataFrame:")
print(platform_a_df_anon.head(20))

In [ ]:
print("\nAnonymised Platform B DataFrame:")
print(platform_b_df_anon.head(20))

In [ ]:
# Use the anonymised data sets as the main data sets for analysis and topic modelling
platform_a_df = platform_a_df_anon
platform_b_df = platform_b_df_anon

<div style="background-color: yellow; padding: 10px;">

<mark>**From this point onwards, analyses and topic modelling are conducted using the anonymised data sets.**</mark>

</div>

In [ ]:
# Criterion 2.8: Find the frequency distribution of the words from each data set's reviews separately
platform_a_words = ' '.join(platform_a_df['Comment']).split() # combines all comments in the platform_a data set into one large string, separated by space, then splits the combined string into individual words based on spaces, creating a list of words
platform_b_words = ' '.join(platform_b_df['Review Content']).split() # as above
platform_a_word_freq = Counter(platform_a_words)  # This counts the occurrence of each word in the Platform A reviews, creating a frequency distribution similar to nltk.FreqDist
platform_b_word_freq = Counter(platform_b_words)
print("Top 10 most common words in Platform A reviews:", platform_a_word_freq.most_common(10))
print()
print("Top 10 most common words in Platform B reviews:", platform_b_word_freq.most_common(10))

In [ ]:
# Criterion 2.9: Plot a histogram/bar plot showing the top 10 words from each data set
def plot_top_words(counter, title):
    words, counts = zip(*counter.most_common(10))
    plt.figure(figsize=(10, 6), facecolor=WHITE)
    sns.barplot(x=list(counts), y=list(words), color=NAVY_BLUE)
    plt.title(title, color=NAVY_BLUE, fontsize=14)
    plt.xlabel('Frequency', color=NAVY_BLUE)
    plt.ylabel('Words', color=NAVY_BLUE)
    plt.gca().set_facecolor(WHITE)
    plt.gca().tick_params(axis='x', colors=NAVY_BLUE)
    plt.gca().tick_params(axis='y', colors=NAVY_BLUE)
    plt.show()

# The plots highlight dominant themes for business insights
plot_top_words(platform_a_word_freq, 'Top 10 Words in Platform A reviews')
plot_top_words(platform_b_word_freq, 'Top 10 Words in Platform B Reviews')

In [ ]:
# Criterion 2.10: Use the word cloud library on the cleaned data and plot the word cloud
def plot_wordcloud(text, title):
    wordcloud = WordCloud(width=800, height=400, background_color=WHITE, color_func=lambda *args, **kwargs: NAVY_BLUE).generate(text) # Word Cloud object is instantiated with custom parameters
    plt.figure(figsize=(10, 5), facecolor=WHITE)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(title, color=NAVY_BLUE, fontsize=14)
    plt.axis('off')
    plt.show()

plot_wordcloud(' '.join(platform_a_df['Comment']), 'Platform A reviews Word Cloud')
plot_wordcloud(' '.join(platform_b_df['Review Content']), 'Platform B Reviews Word Cloud')

In [ ]:
# Criterion 2.11: Create a new dataframe by filtering out the data to extract only the negative reviews from both data sets
platform_a_negative = platform_a_df[platform_a_df['Overall Score'] < 3]  # Platform A reviews, overall scores < 3
platform_b_negative = platform_b_df[platform_b_df['Review Stars'] < 3]  # Platform B reviews, stars < 3
print("Platform A Negative Reviews:", len(platform_a_negative))
print("Platform B Negative Reviews:", len(platform_b_negative))

In [ ]:
# Criterion 2.12: Repeat the frequency distribution and word cloud steps on the filtered data consisting of only negative reviews
platform_a_neg_words = ' '.join(platform_a_negative['Comment']).split()
platform_b_neg_words = ' '.join(platform_b_negative['Review Content']).split()
platform_a_neg_freq = Counter(platform_a_neg_words)
platform_b_neg_freq = Counter(platform_b_neg_words)
plot_top_words(platform_a_neg_freq, 'Top 10 Words in Platform A Negative Reviews')
plot_top_words(platform_b_neg_freq, 'Top 10 Words in Platform B Negative Reviews')
plot_wordcloud(' '.join(platform_a_negative['Comment']), 'Platform A Negative Reviews Word Cloud')
plot_wordcloud(' '.join(platform_b_negative['Review Content']), 'Platform B Negative Reviews Word Cloud')

 **-------------------------------------------------------------------- END OF SECTION----------------------------------------------------------------------- ################################################################################################################################################################################################################**

### **3. Conducting initial topic modelling**

In [ ]:
### Conducting initial topic modelling

# Criterion 3.1: Filter out the reviews that are from the locations common to both data sets (negative reviews dataframes) and merge into a new list
common_neg_platform_a = platform_a_negative[platform_a_negative["Club's Name"].isin(common_locations)] # filters the platform_a negative reviews where the "Club's Name" is in common_locations
common_neg_platform_b = platform_b_negative[platform_b_negative['Location Name'].isin(common_locations)] # as above but for Platform B + 'Location Name'
combined_common_neg = list(common_neg_platform_a['Comment']) + list(common_neg_platform_b['Review Content']) # After filtering, the comments from the two data sets are combined into a single list

In [ ]:
# Criterion 3.2: Preprocess this data set and use BERTopic on this cleaned data set
combined_common_neg = [preprocess_text(text) for text in combined_common_neg] # Each review in the combined_common_neg list is cleaned using preprocess_text(text)
combined_common_neg_subset = combined_common_neg[:1000]  # To speed up computation, only the first 1000 reviews are selected
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device='cpu') # all-MiniLM-L6-v2 is used to generate numerical embeddings for each review
embeddings = embedding_model.encode(combined_common_neg_subset, show_progress_bar=True) # converts the text into high-dimensional vectors
# Note: all-MiniLM-L6-v2 balances efficiency and quality for embedding generation

In [ ]:
# Creating a BERTopic Model
# BERTopic leverages contextual embeddings, outperforming other methods for nuanced topic extraction from negative reviews
topic_model = BERTopic(
    umap_model=umap.UMAP(n_neighbors=30, n_components=2, min_dist=0.0, metric='cosine'), # Configures UMAP for dimensionality reduction with specified parameters
    hdbscan_model=KMeans(n_clusters=10), # KMeans clustering to group data into 10 topics
    embedding_model=None,
    language="english", # sets the language of the text data
    calculate_probabilities=True, # calculates probabilities for topic assignment
    verbose=True
)

# Hyperparameters selection
    # n_neighbors=30, n_components=2 in UMAP reduce dimensionality effectively for clustering.
    # In this case, using n_neighbors=30 enabled the UMAP model to maintain a more generalised understanding of the data set's structure
    # n_clusters=10 as per requiremnt

# Model training
topics, probs = topic_model.fit_transform(combined_common_neg_subset, embeddings=embeddings) # topics lists where each element is a topic label and probs lists each element in a probability indicating the strength of the association with each topic

In [ ]:
# Criterion 3.3: List out the top topics along with their document frequencies
topic_info = topic_model.get_topic_info() # returns a pandas dataframe that contains metadata about all the topics identified by the BERTopic model
print("Top Topics with Document Frequencies:")
print(topic_info.head(10)) # displays the first 10 rows of the dataframe, which represent the top 10 topics

The output is organised in descending order by document count, with the most frequent topics listed first. **A summary of the output is provided in the table below:**

In [ ]:
# Criterion 3.4: For the top 2 topics, list out the top words
for i in range(2):
    print(f"Top words for Topic {i}:", topic_model.get_topic(i))

**Top 2 Topics - List of Top Words**
* Topic 0: Showers & Cleanliness - "showers," "changing," "dirty," "air" signal hygiene issues.
* Topic 1: Equipment & Machines - "equipment," "machines," "one," "broken" point to maintenance woes.

In [ ]:
# # Criterion 3.5: Show an interactive visualisation of the topics (intertopic distance map)
topic_model.visualize_topics().show()

**Intertopic Distance Map**

* Topics 0, 1, 3, 4, and 7 tightly clustered, facility and equipment focus
* Topics 3, 6, 8 (staff/membership/access - near, staff and access themes
* Topics 2 and 7 overlap, amenities and operations
* Topic 9 and 6 (parking and access) overlap distinct parking issues

In [ ]:
# Criterion 3.6: Show a barchart of the topics, displaying the top 5 words in each topic
topic_model.visualize_barchart(top_n_topics=10, n_words=5, custom_labels=True, title="Top Topics Bar Chart").show()

**Top Topics Bar Chart**

* Topic 0 (showers, changing, dirty, air, smell)
* Topic 1 (equipment, machines, one, broken, staff)
* Topic 2 (weights, machines, equipment, people, use)
* Topic 3 (staff, manager, rude, people, name)
* Topic 4 (music, class, classes, people, equipment)

Topics cover cleanliness issues, equipment maintenance, usage concerns, staff interactions, and class experiences.

In [ ]:
# # Criterion 3.7: Plot a heatmap, showcasing the similarity matrix
topic_model.visualize_heatmap().show()

In [ ]:
# Criterion 3.8: For 10 clusters, provide a brief description of the topics
for i in range(min(11, len(topic_info) - 1)):
    topic_words = [word for word, _ in topic_model.get_topic(i)]
    print(f"Topic {i}: General Theme - {topic_info['Name'][i+1]}")
    print(f"Top Words: {', '.join(topic_words[:5])}")
    print(f"Description: Likely reflects {', '.join(topic_words[:3])} concerns.\n")

 **-------------------------------------------------------------------- END OF SECTION----------------------------------------------------------------------- ################################################################################################################################################################################################################**

### **4. Performing further data investigation**

In [ ]:
### Performing further data investigation

# Criterion 4.1: List out the top 20 locations with the highest number of negative reviews separately for Platform A and Platform B
top_20_platform_a_neg = platform_a_negative["Club's Name"].value_counts().head(20)
top_20_platform_b_neg = platform_b_negative['Location Name'].value_counts().head(20)
print("Top 20 Platform A Negative Locations:\n", top_20_platform_a_neg)
print()
print("Top 20 Platform B Negative Locations:\n", top_20_platform_b_neg)
print("Analysis to determine if locations are similar across both data sets.")

In [ ]:
# Criterion 4.2: Merge the 2 data sets using Location Name and Club's Name
# Goal -> Identify high-impact locations, crucial for targeted improvements
platform_a_counts = platform_a_df.groupby("Club's Name")['Comment'].count().reset_index(name='Platform A reviews') # Groups platform_a_df by each club name, counts how many comments exist for each club and resets the index to turn the result into a dataframe
platform_b_counts = platform_b_df.groupby('Location Name')['Review Content'].count().reset_index(name='Platform B Reviews') # as above but for platform_b_df
review_counts = pd.merge(platform_a_counts, platform_b_counts, left_on="Club's Name", right_on='Location Name', how='outer') # Combines the Platform A and Platform B counts into one dataframe
review_counts['Platform A reviews'] = review_counts['Platform A reviews'].fillna(0) # Replaces any missing values with zeros
review_counts['Platform B Reviews'] = review_counts['Platform B Reviews'].fillna(0) # as above
review_counts['Total Reviews'] = review_counts['Platform A reviews'] + review_counts['Platform B Reviews'] # Creates a new column that sums up both Platform A and Platform B reviews and gives the total number of reviews for each club across both platforms

In [ ]:
review_counts.info()

In [ ]:
review_counts.head(20)

In [ ]:
# 4.3    Now, list out the following:
#           • Locations
#           • Number of Platform B reviews for this location
#           • Number of Platform A reviews for this location
#           • Total number of reviews for this location (sum of Platform A reviews and Platform B reviews)

In [ ]:
# Plot list of locations of 'Total Reviews'
review_counts = review_counts.sort_values('Total Reviews', ascending=False)
plt.figure(figsize=(15, 6), facecolor=WHITE)
sns.barplot(x="Club's Name", y='Total Reviews', data=review_counts.sort_values('Total Reviews', ascending=False).head(21), color=NAVY_BLUE)
plt.title('Total Reviews per Location', color=NAVY_BLUE)
plt.xlabel('Location Name', color=NAVY_BLUE)
plt.ylabel('Total Reviews', color=NAVY_BLUE)
plt.gca().set_facecolor(WHITE)
plt.gca().tick_params(axis='x', colors=NAVY_BLUE)
plt.gca().tick_params(axis='y', colors=NAVY_BLUE)
plt.xticks(rotation=90)

In [ ]:
# Plot of number of Platform B reviews per location
review_counts = review_counts.sort_values('Platform B Reviews', ascending=False)
plt.figure(figsize=(15, 6), facecolor=WHITE)
sns.barplot(x="Club's Name", y='Platform B Reviews', data=review_counts.sort_values('Platform B Reviews', ascending=False).head(21), color=NAVY_BLUE)
plt.title('Platform B Reviews per Location', color=NAVY_BLUE)
plt.xlabel('Location Name', color=NAVY_BLUE)
plt.ylabel('Platform B Reviews', color=NAVY_BLUE)
plt.gca().set_facecolor(WHITE)
plt.gca().tick_params(axis='x', colors=NAVY_BLUE)
plt.gca().tick_params(axis='y', colors=NAVY_BLUE)
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Plot of number of Platform A reviews per location
review_counts = review_counts.sort_values('Platform A reviews', ascending=False)
plt.figure(figsize=(15, 6), facecolor=WHITE)
sns.barplot(x="Club's Name", y='Platform A reviews', data=review_counts.sort_values('Platform A reviews', ascending=False).head(21), color=NAVY_BLUE)
plt.title('Platform A reviews per Location', color=NAVY_BLUE)
plt.xlabel('Location Name', color=NAVY_BLUE)
plt.ylabel('Platform A reviews', color=NAVY_BLUE)
plt.gca().set_facecolor(WHITE)
plt.gca().tick_params(axis='x', colors=NAVY_BLUE)
plt.gca().tick_params(axis='y', colors=NAVY_BLUE)
plt.xticks(rotation=90)

In [ ]:
# Criterion 4.4: Sort based on the total number of reviews
review_counts = review_counts.sort_values('Total Reviews', ascending=False) # Sorts the combined dataframe in descending order of total reviews, most reviewed clubs at the top

# Handle unmatched Platform B locations
unmatched_platform_b = platform_b_df[~platform_b_df['Location Name'].isin(platform_a_df["Club's Name"])].groupby('Location Name')['Review Content'].count() # filters (with ~)  to find platform_b locations that don't exist in the Platform A data
unmatched_counts = unmatched_platform_b.reset_index().rename(columns={'Review Content': 'Platform B Reviews', 'Location Name': "Club's Name"}) # converts the series to a dataframe, renames colums to match the structure of the main review_counts dataframe
unmatched_counts['Platform A reviews'] = 0 # as these locations don't appear in Platform A data, sets their Platform A review count to 0
unmatched_counts['Total Reviews'] = unmatched_counts['Platform B Reviews'] # The total reviews equal just the Platform B reviews for these locations
review_counts = pd.concat([review_counts, unmatched_counts], ignore_index=True).sort_values('Total Reviews', ascending=False) # Appends the unmatched locations to the main dataframe and resets index

# This code ensures all locations are ranked by total reviews, combining Platform A
# and Platform B data. First, it sorts matched locations by their total reviews
# Then, it identifies unmatched Platform B locations, assigns them zero Platform A
# reviews, and calculates their total reviews.
# Finally, it appends these unmatched locations to the main data set, re-sorts,
# and displays the top 30 locations for ranking clarity

# Criterion 4.3: List out Locations, Number of Platform B reviews, Number of Platform A reviews, Total number of reviews
print("Top 30 locations (including unmatched Platform B counts):\n", review_counts.head(30))

**Consolidating the information above into a table:**

In [ ]:
# Criterion 4.5: For the top 30 locations, redo the word frequency and word cloud
top_30_locations = review_counts.head(30)["Club's Name"].tolist()
top_30_platform_a = platform_a_df[platform_a_df["Club's Name"].isin(top_30_locations)]['Comment']
top_30_platform_b = platform_b_df[platform_b_df['Location Name'].isin(top_30_locations)]['Review Content']
top_30_combined = pd.concat([top_30_platform_a, top_30_platform_b]).tolist()
top_30_words = ' '.join(top_30_combined).split()
top_30_freq = Counter(top_30_words)
plot_top_words(top_30_freq, 'Top 10 Words in Top 30 Locations')
plot_wordcloud(' '.join(top_30_combined), 'Top 30 Locations Word Cloud')
print("Compare with initial run to identify shifts in focus.")

In [ ]:
# Criterion 4.6: For the top 30 locations, combine the reviews and run them through BERTopic
top_30_embeddings = embedding_model.encode(top_30_combined, show_progress_bar=True) # pre-trained embedding model to convert each review into a numerical vector
top_30_topic_model = BERTopic( # Creats a fresh BERTopic instance specifically for the top 30 locations
    hdbscan_model=KMeans(n_clusters=10),
    embedding_model=None,
    language="english", # Configures for english language tex
    calculate_probabilities=True, #enables probability calculations for topic assignments
    verbose=True
)
# Fitting and transforming the data
top_30_topics, _ = top_30_topic_model.fit_transform(top_30_combined, embeddings=top_30_embeddings) # Trains the model on the reviews from top 30 locations
print("Top Topics for Top 30 Locations:")
print(top_30_topic_model.get_topic_info().head(10))

**Consolidating the information above into a table:**

In [ ]:
# Criterion 4.7: Comment on differences from the first BERTopic run and additional insights
print("Comment: Compare with initial BERTopic run for differences and insights.")

**Comparison Table**

 **-------------------------------------------------------------------- END OF SECTION----------------------------------------------------------------------- ################################################################################################################################################################################################################**

### **5. Conducting emotion analysis**

In [ ]:
### Conducting emotion analysis

# Criterion 5.1: Import the BERT model bhadresh-savani/bert-base-uncased-emotion from Hugging Face and set up a pipeline
emotion_classifier = pipeline("text-classification", model="bhadresh-savani/bert-base-uncased-emotion", top_k=None, device=-1) # creates a hugging face pipeline specifically for text classification tasks
# device=-1 (CPU) due to resource constraints

# model="bhadresh-savani/bert-base-uncased-emotion" - This particular model is fine-tuned to detect emotions in text

In [ ]:
# Criterion 5.2: Run the model with an example sentence and display emotion classifications
example = "I’m furious about the dirty changing rooms!"
print("Example Emotion Classification:", emotion_classifier(example))

In [ ]:
# Criterion 5.3: Run this model on both data sets and capture the top emotion for each review
def get_top_emotion(text): # takes a text string as input
    result = emotion_classifier(text)[0] # Passes it through the emotion classifier model
    return max(result, key=lambda x: x['score'])['label'] # uses max() with a lambda function to find the emotion with the highest score and returns just the emotion label

platform_a_neg_emotions = platform_a_negative['Comment'].apply(get_top_emotion) # takes each negative comment from the Platform A data set and applies the get_top_emotion function to identify the primary emotion
platform_b_neg_emotions = platform_b_negative['Review Content'].apply(get_top_emotion) # as above but for platform_b_df

In [ ]:
# Print results
print("Platform A Negative Emotions:\n", platform_a_neg_emotions.value_counts())
print()
print("Platform B Negative Emotions:\n", platform_b_neg_emotions.value_counts())

In [ ]:
# Criterion 5.4: Use a bar plot to show the top emotion distribution for all negative reviews
plt.figure(figsize=(10, 6), facecolor=WHITE)
sns.countplot(x=pd.concat([platform_a_neg_emotions, platform_b_neg_emotions]), color=DARK_RED)
plt.title('Top Emotion Distribution in Negative Reviews', color=NAVY_BLUE)
plt.xlabel('Emotion', color=NAVY_BLUE)
plt.ylabel('Count', color=NAVY_BLUE)
plt.gca().set_facecolor(WHITE)
plt.gca().tick_params(axis='x', colors=NAVY_BLUE)
plt.gca().tick_params(axis='y', colors=NAVY_BLUE)
plt.show()

In [ ]:
# Criterion 5.5: Extract all the negative reviews where anger is the top emotion
anger_reviews = pd.concat([platform_a_negative[platform_a_neg_emotions == 'anger']['Comment'], # Filters Platform A and platform_b negative reviews to keep only those where 'anger' is the dominant emotion
                           platform_b_negative[platform_b_neg_emotions == 'anger']['Review Content']]).tolist()

In [ ]:
# Criterion 5.6: Run BERTopic on the anger reviews
anger_embeddings = embedding_model.encode(anger_reviews, show_progress_bar=True) # Transforms each anger review into a numerical vector representation
anger_topic_model = BERTopic( # Creates a new BERTopic model instance specifically for anger reviews
    hdbscan_model=KMeans(n_clusters=10), # ues KMeans clustering with exactly 10 clusters
    embedding_model=None,
    language="english",
    calculate_probabilities=True,
    verbose=True
)

In [ ]:
# BERTopic uses prior settings (n_clusters=10), ensuring consistency

# Running the Topic model
anger_topics, _ = anger_topic_model.fit_transform(anger_reviews, embeddings=anger_embeddings) # Trains the model on the anger reviews, returns topic assignments (ignoring probabilities with _)
print("Top Topics for Anger Reviews:")
print(anger_topic_model.get_topic_info().head(10))

**Summary of the output is provided in the table below**

In [ ]:
# Criterion 5.7: Visualise the clusters from this run and comment on differences
anger_topic_model.visualize_topics().show()
print("Comment: Assess differences from previous runs to pinpoint anger triggers.")

**Comparison Table**

**Side by Side Intertopic Distance Maps**

In [ ]:
# Rendering the "Side by Side Intertopic Distance Maps" image
import IPython.display as display
from PIL import Image
import requests
from io import BytesIO

# Open and display the PNG image
image_path = r"data/reviews_file1.xlsx"
response = requests.get(image_path) # Sends a GET request to the URL to download the image
img = Image.open(BytesIO(response.content)) # Opens the downloaded image
display.display(img)

 **-------------------------------------------------------------------- END OF SECTION----------------------------------------------------------------------- ################################################################################################################################################################################################################**

### **6. Using a large language model from Hugging Face**

In [ ]:
# 6. Using a large language model from Hugging Face

# Suppress logging noise
logging.getLogger("transformers").setLevel(logging.ERROR) # only display ERROR and CRITICAL level messages from the "transformers" library

# Verify data
print("Checking anger_reviews:", type(anger_reviews), anger_reviews[:5])

In [ ]:
# Criterion 6.1: Load Falcon-7B-Instruct
torch.cuda.empty_cache() # clears the GPU memory cache in PyTorch
falcon = pipeline("text-generation", model="tiiuae/falcon-7b-instruct", device=-1) # creates a text generation pipeline
print("Model loaded successfully")

<div style="background-color: yellow; padding: 10px;">

<mark>**The code below uses the first 50 reviews from anger_reviews to create a smaller, manageable subset for analysis, to improve efficiency rather than processing the entire data set.**</mark>
</div>

In [ ]:
# Criteria 6.2–6.4: Process reviews
subset_anger_reviews = anger_reviews[:50] # Takes the first 50 reviews from anger_reviews
prompt = ( # template prompt that will be used with each review,  it instructs the model to analyse a customer review and identify exactly 3 main topics
    "Analyse the following customer review and identify exactly 3 main topics. "
    "Return them strictly in this exact format, with no additional text or deviation:\n"
    "1. [Topic]\n2. [Topic]\n3. [Topic]\n\n"
    "Review:\n"
)

# Please note that I had to tweak the prompt as I was getting very poor results
# with the original prompt, which was: "In the following customer review, pick out the main 3 topics. Return them in a numbered list format, with each one on a new line."

falcon_outputs = [] # Disclosure: parts of this code were created with AI's assistance
for i, review in enumerate(subset_anger_reviews): # iterates through each review in the subset, for each review, it combines the template prompt with the specific review text
    output = falcon( # calls the Falcon model with defined parameters
        prompt + review,
        max_new_tokens=500, # Limits the response to 500 new tokens (~ 375-500 words)
        temperature=0.7, # Sets moderate randomness in the generation (higher values = more creative/random)
        top_p=0.9, # Uses nucleus sampling, considering only the top 90% most probable next tokens
        do_sample=True # Enables probabilistic sampling rather than just taking the most likely token
    )[0]['generated_text']
    falcon_outputs.append(output)

    # Hyperparameters selection - after multiple trial and errors, and with AI's help, the hyperparameters were set as follows:
    # max_new_tokens=500: Limits output length for efficiency while ensuring detail
    # temperature=0.7, top_p=0.9: Balances creativity and coherence in topic/insight generation

In [ ]:
# Criterion 6.5: Topic extraction - Disclosure: parts of this code were created with AI's assistance
all_topics = [] # empty list to store all extracted topics
for i, output in enumerate(falcon_outputs): # Iterates through each model output with its index
    print(f"Raw Output {i+1}:", output) # Prints the raw output for reference
    lines = output.split('\n') # Splits the output into lines
    review_idx = next((idx for idx, line in enumerate(lines) if "Review:" in line), -1) # finds where the review text begins by locating the line containing "Review:"
    content_lines = lines[review_idx + 1:] if review_idx != -1 else lines #  isolates just the content after the "Review:" marker
    topics = re.findall(r'^\d+\.\s*\[?(.+?)\]?\s*$', '\n'.join(content_lines), re.MULTILINE) # find numbered lines
    topics = [t for t in topics if t.lower() != "topic"]
    if not topics:
        review = subset_anger_reviews[i]
        keywords = re.findall(r'\b(?:staff|equipment|service|crowd|hygiene|clean|rude|machine)\w*\b', review, re.IGNORECASE)
        topics = list(dict.fromkeys(keywords))[:3]
    if len(topics) < 3:
        topics.extend([f"Concern {j+1} from review {i+1}" for j in range(3 - len(topics))])
    print(f"Extracted Topics {i+1}: {topics[:3]}")
    all_topics.extend(topics[:3])
print("All Topics:", all_topics)
print("Sample Extracted Topics:", all_topics[:10])
if not all_topics:
    raise ValueError("No topics extracted - extraction logic failed")

# This code extracts topics from falcon_outputs into all_topics, looping through
# each output to print raw text, split it into lines, and isolate content after
# "Review:". It uses regex to find numbered topics, or extracts up to three
# keywords (e.g., "staff," "equipment") if none are found, adding placeholders
# if needed. Topics are printed, added to all_topics, and the code prints all
# topics plus a sample of the first 10, raising an error if no topics are extracted

In [ ]:
# Criterion 6.6: Run BERTopic
topics_model = BERTopic(
    hdbscan_model=KMeans(n_clusters=10),
    embedding_model=SentenceTransformer("all-MiniLM-L6-v2", device='cpu'),
    language="english",
    calculate_probabilities=True,
    verbose=True
)
topics, _ = topics_model.fit_transform(all_topics)
print("BERTopic on Falcon Topics:")
print(topics_model.get_topic_info().head(10))

# Criterion 6.7: Comment
print("Comment: BERTopic clusters refine anger-driven topics into categories like staff issues, equipment problems, and facility conditions.")

In [ ]:
# Criteria 6.8–6.11: Generate insights - # Disclosure: parts of this code were created with AI's assistance
insight_prompt = (
    "Given the following topic cluster from negative customer reviews, provide exactly 3 actionable insights "
    "for the wellness centre company. Return them strictly in this format, with no additional text:\n"
    "1. [Topic] - [Suggestion]\n"
    "2. [Topic] - [Suggestion]\n"
    "3. [Topic] - [Suggestion]\n\n"
    "Cluster:\n"
)

# Please note that here I also had to modify the prompt as I was getting unsuccessful results
# Original prompt: "For the following text topics obtained from negative customer reviews, can you give some actionable insights that would help this wellness centre company?"

insights = []
cluster_info = topics_model.get_topic_info()
for _, row in cluster_info.iterrows():
    cluster_name = row['Name']
    output = falcon(
        insight_prompt + cluster_name,
        max_new_tokens=500,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )[0]['generated_text']
    print(f"Cluster '{cluster_name}' Output:", output)
    # Relaxed parsing: match numbered lines, assume content is suggestion
    lines = output.split('\n')
    review_idx = next((idx for idx, line in enumerate(lines) if "Cluster:" in line), -1)
    content_lines = lines[review_idx + 1:] if review_idx != -1 else lines
    numbered_lines = [line.strip() for line in content_lines if line.strip().startswith(('1.', '2.', '3.'))]
    # Fallback: if no numbered lines, use raw lines and pair with cluster topics
    if not numbered_lines:
        cluster_topics = cluster_name.split('_')[1:]  # Extract topics from cluster name
        numbered_lines = [f"{i+1}. [{cluster_topics[0] if i < len(cluster_topics) else 'General Issue'}] - {line.strip()}"
                         for i, line in enumerate(content_lines[:3]) if line.strip()]
    insights.extend(numbered_lines[:3])

**Summary Table**

In [ ]:
# Criterion 6.11: List insights
print("Actionable Insights:")
for insight in insights:
    print(insight)
if not insights:
    print("Warning: No insights generated - verify Falcon output")

 **-------------------------------------------------------------------- END OF SECTION----------------------------------------------------------------------- ################################################################################################################################################################################################################**

### **7. Using Gensim**

In [ ]:
### Using Gensim

# Criterion 7.1: Perform the preprocessing required to run the LDA model from Gensim (using negative reviews)
neg_reviews = pd.concat([platform_a_negative['Comment'], platform_b_negative['Review Content']]).tolist()
tokenized_reviews = [word_tokenize(review) for review in neg_reviews]
dictionary = corpora.Dictionary(tokenized_reviews)
corpus = [dictionary.doc2bow(text) for text in tokenized_reviews]

# THe code preprocesses negative reviews for LDA by combining Platform A and
# Platform B comments into neg_reviews, tokenising them into tokenized_reviews,
# creating a word-to-ID dictionary, and forming a BoW corps

In [ ]:
# Criterion 7.2: Using Gensim, perform LDA on the tokenised data with 10 topics
# LDA offers a statistical alternative to BERTopic, validating findings via a different approach
lda_model = LdaModel(corpus, num_topics=10, id2word=dictionary, passes=15)

# Hyperparameters
    # num_topics=10: meets project specification and prior clustering
    # passes=15: Ensures convergence for robust topic extraction

# The code trains an LDA model with Gensim on a BoW corpus negative
# reviews, extracting 10 topics over 15 passes using the preprocessed
# dictionary. It aims to identify themes like hygiene or staff issues,
# complementing prior BERTopic and Falcon analysess.

In [ ]:
# Criterion 7.3: Show the visualisations of the topics (distance maps and bar chart via pyLDAvis)
vis_data = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.display(vis_data)
# pyLDAvis aids interpretability

# The code uses gensimvis.prepare and pyLDAvis.display to create an interactive
# LDA visualisation from the trained model (10 topics), BoW corpus,
# and dictionary

In [ ]:
# Criterion 7.4: Comment on the output and compare with other techniques
print("Comment: Compare LDA topics with BERTopic for consistency and additional insights.")

<div style="background-color: yellow; padding: 10px;">

### <mark><font color="#8A0707"><b>**8. Final Comparison Table of NLP Topic Modelling Results**</b></font></mark>

</div>

 **-------------------------------------------------------------------- END OF CODE -------------------------------------------------------------------------- ################################################################################################################################################################################################################**

### **Code Summary**

In [ ]:
# Track variables
%whos

**Summary Table**

In [ ]:
!pip install pdfplumber

**Checking Word count in report file (.pdf)**